# Phase 3 — Model Training

**Project:** AI Mental Health Assessment Platform  
**Goal:** Train 3 machine learning models from the same dataset:

1. Predict `mental_health_condition`
2. Predict `severity`
3. Predict `treatment`

This notebook loads the processed train/test datasets from Phase 2, trains multiple classification models, compares performance, tunes the best models, and saves final models for the next phases.

## Step 1 — Import Libraries

We import core libraries, machine learning models, evaluation metrics, and save/load utilities.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Optional models: notebook will run even if these libraries are not installed.
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except Exception:
    LIGHTGBM_AVAILABLE = False

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception:
    CATBOOST_AVAILABLE = False

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## Step 2 — Define Project Paths

The notebook is designed for your GitHub repo structure.

Expected structure:

```text
ai-mental-health-assessment-platform/
├── data/
│   ├── mental_health_prediction.csv
│   └── processed/
├── notebooks/
├── models/
├── reports/
└── src/
```

In [ ]:
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = ROOT_DIR / "models"
REPORTS_DIR = ROOT_DIR / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR:", ROOT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

## Step 3 — Target Configuration

We train 3 separate models, one for each target.

In [ ]:
TARGETS = {
    "condition": "mental_health_condition",
    "severity": "severity",
    "treatment": "treatment"
}

TARGETS

## Step 4 — Helper Function: Load Phase 2 Processed Data

Phase 2 should create this structure:

```text
data/processed/condition/X_train.csv
.../X_test.csv
.../y_train.csv
.../y_test.csv
```

If your Phase 2 notebook created these files, this function will load them.

In [ ]:
def load_processed_target_data(target_key):
    target_dir = PROCESSED_DIR / target_key
    files = {
        "X_train": target_dir / "X_train.csv",
        "X_test": target_dir / "X_test.csv",
        "y_train": target_dir / "y_train.csv",
        "y_test": target_dir / "y_test.csv",
    }
    missing_files = [str(path) for path in files.values() if not path.exists()]
    if missing_files:
        raise FileNotFoundError(
            "Phase 2 processed files not found. Please run notebooks/02_data_preprocessing.ipynb first.
"
            + "Missing files:
" + "
".join(missing_files)
        )

    X_train = pd.read_csv(files["X_train"])
    X_test = pd.read_csv(files["X_test"])
    y_train = pd.read_csv(files["y_train"]).squeeze("columns")
    y_test = pd.read_csv(files["y_test"]).squeeze("columns")
    return X_train, X_test, y_train, y_test

## Step 5 — Load All 3 Target Datasets

In [ ]:
data = {}

for target_key in TARGETS.keys():
    X_train, X_test, y_train, y_test = load_processed_target_data(target_key)
    data[target_key] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }
    print(f"{target_key.upper()}")
    print("X_train:", X_train.shape)
    print("X_test :", X_test.shape)
    print("y_train:", y_train.shape)
    print("y_test :", y_test.shape)
    print("Classes:", sorted(y_train.unique()))
    print("-" * 80)

## Step 6 — Build Model Dictionary

We will compare multiple classification models.

If XGBoost, LightGBM, or CatBoost are not installed, the notebook will skip them automatically.

In [ ]:
def get_models():
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        "Extra Trees": ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    }

    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=RANDOM_STATE,
            eval_metric="mlogloss",
            verbosity=0
        )

    if LIGHTGBM_AVAILABLE:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            verbose=-1
        )

    if CATBOOST_AVAILABLE:
        models["CatBoost"] = CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=5,
            random_seed=RANDOM_STATE,
            verbose=False
        )

    return models

models = get_models()
list(models.keys())

## Step 7 — Evaluation Function

This function calculates common classification metrics.

In [ ]:
def evaluate_classifier(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_Macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall_Macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1_Macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "Precision_Weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall_Weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1_Weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }

## Step 8 — Train Models for One Target

This function trains all models for a single target and returns comparison results.

In [ ]:
def train_models_for_target(target_key, X_train, X_test, y_train, y_test):
    results = []
    fitted_models = {}

    model_dict = get_models()

    for model_name, model in model_dict.items():
        print(f"Training {target_key} → {model_name}...")
        start_time = time.time()

        try:
            model.fit(X_train, y_train)
            metrics = evaluate_classifier(model, X_test, y_test)
            training_time = time.time() - start_time

            result = {
                "Target": target_key,
                "Model": model_name,
                **metrics,
                "Training_Time_Seconds": round(training_time, 3)
            }
            results.append(result)
            fitted_models[model_name] = model

        except Exception as e:
            print(f"Skipped {model_name} due to error: {e}")

    results_df = pd.DataFrame(results).sort_values("F1_Weighted", ascending=False)
    return results_df, fitted_models

## Step 9 — Train Models for Target 1: Mental Health Condition

In [ ]:
condition_results, condition_models = train_models_for_target(
    "condition",
    data["condition"]["X_train"],
    data["condition"]["X_test"],
    data["condition"]["y_train"],
    data["condition"]["y_test"]
)

condition_results

## Step 10 — Train Models for Target 2: Severity

In [ ]:
severity_results, severity_models = train_models_for_target(
    "severity",
    data["severity"]["X_train"],
    data["severity"]["X_test"],
    data["severity"]["y_train"],
    data["severity"]["y_test"]
)

severity_results

## Step 11 — Train Models for Target 3: Treatment

In [ ]:
treatment_results, treatment_models = train_models_for_target(
    "treatment",
    data["treatment"]["X_train"],
    data["treatment"]["X_test"],
    data["treatment"]["y_train"],
    data["treatment"]["y_test"]
)

treatment_results

## Step 12 — Combine All Model Results

In [ ]:
all_results = pd.concat(
    [condition_results, severity_results, treatment_results],
    ignore_index=True
).sort_values(["Target", "F1_Weighted"], ascending=[True, False])

all_results

## Step 13 — Save Model Comparison Results

In [ ]:
condition_results.to_csv(REPORTS_DIR / "condition_model_results.csv", index=False)
severity_results.to_csv(REPORTS_DIR / "severity_model_results.csv", index=False)
treatment_results.to_csv(REPORTS_DIR / "treatment_model_results.csv", index=False)
all_results.to_csv(REPORTS_DIR / "all_model_comparison.csv", index=False)

print("Saved model comparison CSV files inside reports/ folder.")

## Step 14 — Visualize Model Comparison

In [ ]:
def plot_model_comparison(results_df, target_key):
    plt.figure(figsize=(12, 6))
    plt.bar(results_df["Model"], results_df["F1_Weighted"])
    plt.title(f"Model Comparison for {target_key}: Weighted F1 Score")
    plt.xlabel("Model")
    plt.ylabel("Weighted F1 Score")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

plot_model_comparison(condition_results, "Mental Health Condition")
plot_model_comparison(severity_results, "Severity")
plot_model_comparison(treatment_results, "Treatment")

## Step 15 — Select Best Baseline Models

We select the best model by `F1_Weighted` because it works well for multiclass classification and handles class imbalance better than accuracy alone.

In [ ]:
def get_best_model(results_df, fitted_models):
    best_model_name = results_df.iloc[0]["Model"]
    best_model = fitted_models[best_model_name]
    best_score = results_df.iloc[0]["F1_Weighted"]
    return best_model_name, best_model, best_score

best_condition_name, best_condition_model, best_condition_score = get_best_model(condition_results, condition_models)
best_severity_name, best_severity_model, best_severity_score = get_best_model(severity_results, severity_models)
best_treatment_name, best_treatment_model, best_treatment_score = get_best_model(treatment_results, treatment_models)

print("Best condition model:", best_condition_name, best_condition_score)
print("Best severity model :", best_severity_name, best_severity_score)
print("Best treatment model:", best_treatment_name, best_treatment_score)

## Step 16 — Hyperparameter Tuning Function

For simplicity and speed, we tune Random Forest, Extra Trees, and Logistic Regression.  
If your best baseline model is different, this function will still choose a safe tuning grid.

In [ ]:
def get_param_grid(model_name):
    if model_name in ["Random Forest", "Extra Trees"]:
        return {
            "n_estimators": [100, 300, 500],
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2]
        }

    if model_name == "Decision Tree":
        return {
            "max_depth": [None, 3, 5, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }

    if model_name == "Logistic Regression":
        return {
            "C": [0.01, 0.1, 1, 10],
            "solver": ["lbfgs"],
            "max_iter": [2000]
        }

    if model_name == "Gradient Boosting":
        return {
            "n_estimators": [100, 200],
            "learning_rate": [0.03, 0.05, 0.1],
            "max_depth": [2, 3, 5]
        }

    # For optional boosting libraries, use a small safe grid.
    if model_name in ["XGBoost", "LightGBM", "CatBoost"]:
        return {
            "n_estimators" if model_name != "CatBoost" else "iterations": [100, 300],
            "learning_rate": [0.03, 0.05, 0.1]
        }

    return {}


def tune_best_model(model_name, base_model, X_train, y_train):
    param_grid = get_param_grid(model_name)

    if not param_grid:
        print(f"No tuning grid found for {model_name}. Returning baseline model.")
        return base_model, {}

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring="f1_weighted",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)
    return grid.best_estimator_, grid.best_params_

## Step 17 — Tune Best Model for Mental Health Condition

In [ ]:
tuned_condition_model, condition_best_params = tune_best_model(
    best_condition_name,
    best_condition_model,
    data["condition"]["X_train"],
    data["condition"]["y_train"]
)

condition_best_params

## Step 18 — Tune Best Model for Severity

In [ ]:
tuned_severity_model, severity_best_params = tune_best_model(
    best_severity_name,
    best_severity_model,
    data["severity"]["X_train"],
    data["severity"]["y_train"]
)

severity_best_params

## Step 19 — Tune Best Model for Treatment

In [ ]:
tuned_treatment_model, treatment_best_params = tune_best_model(
    best_treatment_name,
    best_treatment_model,
    data["treatment"]["X_train"],
    data["treatment"]["y_train"]
)

treatment_best_params

## Step 20 — Evaluate Tuned Models

In [ ]:
tuned_results = []

for target_key, model, params in [
    ("condition", tuned_condition_model, condition_best_params),
    ("severity", tuned_severity_model, severity_best_params),
    ("treatment", tuned_treatment_model, treatment_best_params),
]:
    metrics = evaluate_classifier(
        model,
        data[target_key]["X_test"],
        data[target_key]["y_test"]
    )
    tuned_results.append({
        "Target": target_key,
        "Best_Model": type(model).__name__,
        **metrics,
        "Best_Params": str(params)
    })

final_results = pd.DataFrame(tuned_results)
final_results

## Step 21 — Classification Reports

In [ ]:
def print_classification_report(target_key, model):
    X_test = data[target_key]["X_test"]
    y_test = data[target_key]["y_test"]
    y_pred = model.predict(X_test)

    print("=" * 100)
    print(f"Classification Report: {target_key.upper()}")
    print("=" * 100)
    print(classification_report(y_test, y_pred, zero_division=0))

print_classification_report("condition", tuned_condition_model)
print_classification_report("severity", tuned_severity_model)
print_classification_report("treatment", tuned_treatment_model)

## Step 22 — Confusion Matrices

In [ ]:
def plot_confusion_matrix(target_key, model):
    X_test = data[target_key]["X_test"]
    y_test = data[target_key]["y_test"]
    y_pred = model.predict(X_test)
    labels = sorted(y_test.unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Confusion Matrix: {target_key}")
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

plot_confusion_matrix("condition", tuned_condition_model)
plot_confusion_matrix("severity", tuned_severity_model)
plot_confusion_matrix("treatment", tuned_treatment_model)

## Step 23 — Feature Importance

Feature importance works for tree-based models. If the model does not support it, the notebook will skip it.

In [ ]:
def plot_feature_importance(target_key, model, top_n=20):
    X_train = data[target_key]["X_train"]

    if not hasattr(model, "feature_importances_"):
        print(f"{target_key}: {type(model).__name__} does not support feature_importances_.")
        return

    importance_df = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": model.feature_importances_
    }).sort_values("Importance", ascending=False).head(top_n)

    plt.figure(figsize=(10, 7))
    plt.barh(importance_df["Feature"][::-1], importance_df["Importance"][::-1])
    plt.title(f"Top {top_n} Feature Importance: {target_key}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

    importance_df.to_csv(REPORTS_DIR / f"{target_key}_feature_importance.csv", index=False)
    return importance_df

condition_importance = plot_feature_importance("condition", tuned_condition_model)
severity_importance = plot_feature_importance("severity", tuned_severity_model)
treatment_importance = plot_feature_importance("treatment", tuned_treatment_model)

## Step 24 — Save Final Models

These files will be used later by FastAPI and Streamlit.

In [ ]:
joblib.dump(tuned_condition_model, MODELS_DIR / "condition_best_model.pkl")
joblib.dump(tuned_severity_model, MODELS_DIR / "severity_best_model.pkl")
joblib.dump(tuned_treatment_model, MODELS_DIR / "treatment_best_model.pkl")

final_results.to_csv(REPORTS_DIR / "final_tuned_model_results.csv", index=False)

print("Saved final models:")
print(MODELS_DIR / "condition_best_model.pkl")
print(MODELS_DIR / "severity_best_model.pkl")
print(MODELS_DIR / "treatment_best_model.pkl")

## Step 25 — Save Model Metadata

This metadata helps future phases know which models were selected.

In [ ]:
model_metadata = {
    "project": "AI Mental Health Assessment Platform",
    "phase": "Phase 3 - Model Training",
    "targets": TARGETS,
    "best_baseline_models": {
        "condition": best_condition_name,
        "severity": best_severity_name,
        "treatment": best_treatment_name,
    },
    "best_parameters": {
        "condition": condition_best_params,
        "severity": severity_best_params,
        "treatment": treatment_best_params,
    },
    "saved_model_files": {
        "condition": "models/condition_best_model.pkl",
        "severity": "models/severity_best_model.pkl",
        "treatment": "models/treatment_best_model.pkl",
    }
}

with open(REPORTS_DIR / "model_training_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=4)

model_metadata

## Step 26 — Create Model Training Report

In [ ]:
report = f"""
# Phase 3 — Model Training Report

## Project
AI Mental Health Assessment Platform

## Goal
Train 3 separate machine learning models from the same lifestyle dataset.

## Targets
1. Mental Health Condition
2. Severity Level
3. Recommended Treatment

## Best Baseline Models
- Condition: {best_condition_name}
- Severity: {best_severity_name}
- Treatment: {best_treatment_name}

## Final Tuned Results

{final_results.to_markdown(index=False)}

## Saved Model Files
- `models/condition_best_model.pkl`
- `models/severity_best_model.pkl`
- `models/treatment_best_model.pkl`

## Next Phase
Phase 4: Model Evaluation with ROC-AUC, learning curves, advanced confusion matrices, and detailed error analysis.
"""

with open(REPORTS_DIR / "model_training_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print(report)

## Step 27 — Final Prediction Test

We test whether the 3 models can predict together.

In [ ]:
sample_condition = data["condition"]["X_test"].iloc[[0]]
sample_severity = data["severity"]["X_test"].iloc[[0]]
sample_treatment = data["treatment"]["X_test"].iloc[[0]]

prediction_output = {
    "mental_health_condition": tuned_condition_model.predict(sample_condition)[0],
    "severity": tuned_severity_model.predict(sample_severity)[0],
    "treatment": tuned_treatment_model.predict(sample_treatment)[0]
}

prediction_output

## Step 28 — Phase 3 Conclusion

Phase 3 is complete.

### Outputs created

```text
models/
├── condition_best_model.pkl
├── severity_best_model.pkl
└── treatment_best_model.pkl

reports/
├── condition_model_results.csv
├── severity_model_results.csv
├── treatment_model_results.csv
├── all_model_comparison.csv
├── final_tuned_model_results.csv
├── model_training_metadata.json
└── model_training_report.md
```

### Next Step
Move to **Phase 4 — Model Evaluation**.